# Notebook API France Travail 

Le but de ce notebook est de présenter les fonctions et méthodes de tri, dans le but de filtrer les offres de la BDD France Travail par critère de recherche.   
Parmi ces critères on trouve : 
 - mots-clefs
 - seniorité
 - distance par rapport à une commune
 - salaire
 - ...

In [15]:
import requests
from datetime import datetime, timedelta, timezone
import re
import json
import os
from dotenv import load_dotenv


load_dotenv()
CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]

## Fonctions de requête et filtre

Date d'aujourd'hui au bon format + Date ultérieur de x jours (format UTC)

In [2]:
def date_moins_x_jours(x: int) -> str:
    date = datetime.now(timezone.utc) - timedelta(days=x)
    return date.strftime("%Y-%m-%dT%H:%M:%SZ")

In [3]:
def maintenant(): 
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

Récupérer le token

In [4]:
def get_token():
    """Récupère un token OAuth2 (valable 30 min)."""
    url = "https://entreprise.francetravail.fr/connexion/oauth2/access_token"
    params = {"realm": "/partenaire"}
    data = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "scope": "api_offresdemploiv2 o2dsoffre",
    }
    r = requests.post(url, params=params, data=data)
    r.raise_for_status()
    return r.json()["access_token"]

Récupérer code géo commune 

In [5]:
token = get_token()
url = "https://api.francetravail.io/partenaire/offresdemploi/v2/referentiel/communes"
headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    }
r = requests.get(url, headers=headers)
data_communes = r.json()
resultats = [item for item in data_communes if item.get('codePostal') == '92130']
resultats
## 92040 pour issy-les-moulineaux


[{'code': '92040',
  'libelle': 'ISSY LES MOULINEAUX',
  'codePostal': '92130',
  'codeDepartement': '92'}]

Requête des emplois par paramètres

In [7]:
def search_jobs(token, 
                keywords, 
                publieeDepuis,
                code_commune_habitat=None, 
                distance=None, 
                departements_search = None, 
                contract_type=None, 
                max_results=150):
 

    url = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    }

    params = {
            "motsCles": keywords,
            "range": f"0-{max_results - 1}",

            "minCreationDate": date_moins_x_jours(publieeDepuis),
            "maxCreationDate": maintenant(), 
            
            "commune": code_commune_habitat,
            "distance": distance,
            "departement": departements_search,

            "typeContrat": contract_type
        }
 
    r = requests.get(url, headers=headers, params=params)
    
    if not r.ok :
        print(f"Erreur {r.status_code} : {r.text}")
        return []
    
    if r.status_code == 204 : 
        print('Aucune annonces pour ces paramètres')
        return []
    
    return r.json().get("resultats", [])


Méthodes de filtre d'emplois avancées

In [14]:
def requires_experience(text: str, max_xp: int, display_info = False) -> bool:
    """
    - text : description/fiche de poste
    - max_xp : seuil maximale d'expérience recherchée : entier strictement positif
    Return true si l'expérience du poste est supérieur à celle de max_xp. (poste trop senior) 
    Exemple : 
        text = 'Niveau d'expérience minimum 3-5 ans'
        Ou 
        text = 'expérience de 10 ans en data analyse'
        max_xp = 2
        return True car l'expérience demandé est supérieur au seuil
    """    
    true_text = text.replace("\n", " ")
    true_text = true_text.replace("\\", "")
    pattern = (
    r"(?:expérience|experience|xp).{0,40}?(\d+)\s*(?:[-àa/]\s*\d+)?\s*[+\-]?\s*(?:ans?|années?|years?)"
    r"|"
    r"(\d+)\s*(?:[-àa/]\s*\d+)?\s*[+\-]?\s*(?:ans?|années?|years?).{0,40}?(?:expérience|experience|xp|minimum|au moins|at least|ship)"
)
    match = re.search(pattern, true_text.lower())
    if match:
        years = int(match.group(1) or match.group(2))
        if display_info : 
            print(match, 'age mini : ', years)
        return  years < 20 and years >= max_xp 
    
    return False


In [15]:
def filter_jobs(jobs, 
                exclude_alternance=True,
                exclude_big_xp=True,
                exclude_keywords=None,
                exclude_xp_description = 2
                ): 
    
    """
    exclude_atlernance : exclusion de alternance + stage via valeur du dic alternance
    exclude_big_xp : Première exclusion de jobs dont l'xp demandé est très elevé, valeur à 4 par defaut (ne pas mettre trop bas car qq faux positifs dans le labels experienceLibelle)
    exclude_keywords : exclusion de mots-clefs present dans l'intitulé du post : senior, lead, technicien,...
    exclude_xp_description : exclusion de l'xp demandé max dans la description via regex
    """
    
    filtered = []
    excluded_log = {"alternance": [], "big_xp": [], "keywords": [], 'xp_desc': []}

    for job in jobs : 

        if exclude_alternance and job['alternance'] :
            excluded_log['alternance'].append(job['origineOffre']['urlOrigine'])
            continue
        
        if exclude_big_xp : 
            xp = job["experienceLibelle"]
            if xp and xp[0].isdigit() and int(xp[0]) >= 4:
                excluded_log['big_xp'].append(job['origineOffre']['urlOrigine'])
                continue

        if exclude_keywords :
            title = job.get("intitule", "").lower()
            if any(kw in title for kw in exclude_keywords):
                excluded_log['keywords'].append(job['origineOffre']['urlOrigine'])
                continue

        if exclude_xp_description : 
            description = job.get("description", "").lower()
            if requires_experience(description, exclude_xp_description) :
                excluded_log['xp_desc'].append(job['origineOffre']['urlOrigine']) 
                continue


        filtered.append(job)

    return excluded_log, filtered
    

Affichage des résultats (jobs filtrés + exclus)

In [16]:
def display_jobs(jobs, excluded=None):
    print("=" * 55)
    print (f'{len(jobs)} jobs trouvés : ')
    for j in jobs : 
         print(f"\n📌 {j.get('intitule', 'N/A')}")
         print(f"   Lieu       : {j.get('lieuTravail', {}).get('libelle', 'N/A')}")
         print(f"   Lien       : {j.get('origineOffre', {}).get('urlOrigine', 'N/A')}")

    if excluded :
         print("=" * 55)
         len_excluded = sum(len(v) for v in excluded.values())
         print (f'{len_excluded} jobs exclus : ')
         print(json.dumps(excluded, indent=2, ensure_ascii=False))

Test d'utilisation

In [19]:
token = get_token()
keywords = "data scientist"
jobs = search_jobs(token=token, keywords=keywords, publieeDepuis=31, code_commune_habitat=92040, distance= 15)

EXCLUSIONS = [
        "senior", "sénior", "sr", "lead", "manager", "directeur", "responsable",
        "confirmé", "expérimenté", "expert", "head of", "chef de projet",
        "alternance", "alternant", "apprenti", "intern",
        "technicien", "cyber", "cloud", "Power BI"
    ]

excluded, filtered = filter_jobs(jobs, exclude_alternance=True, exclude_big_xp=True, exclude_keywords=EXCLUSIONS, exclude_xp_description = 2)
display_jobs(filtered, excluded)


11 jobs trouvés : 

📌 Data Scientist Junior pour projet de recherche en psychiatrie (H/F)
   Lieu       : 75 - PARIS 14
   Lien       : https://candidat.francetravail.fr/offres/recherche/detail/209QGPH

📌 Data Scientist (H/F)
   Lieu       : 92 - Levallois-Perret
   Lien       : https://candidat.francetravail.fr/offres/recherche/detail/208PZDB

📌 Chargé de traitement des données et Référent UNESS (H/F)
   Lieu       : 75 - Paris 12e Arrondissement
   Lien       : https://candidat.francetravail.fr/offres/recherche/detail/209ZNBJ

📌 Actuaire / Data Scientist (H/F)
   Lieu       : 92 - Bois-Colombes
   Lien       : https://candidat.francetravail.fr/offres/recherche/detail/3986725

📌 Data Scientist F/H (H/F)
   Lieu       : 92 - Courbevoie
   Lien       : https://candidat.francetravail.fr/offres/recherche/detail/3884999

📌 Data Scientist / AI Engineer (H/F)
   Lieu       : 92 - Levallois-Perret
   Lien       : https://candidat.francetravail.fr/offres/recherche/detail/3850813

📌 DATA ANALYS

In [13]:
pattern = (
    r"(?:expérience|experience|xp).{0,40}?(\d+)[+\-]?\s*(?:ans?|années?|years?)"
    r"|"
    r"(\d+)[+\-]?\s*(?:ans?|années?|years?).{0,40}?(?:expérience|experience|xp|minimum|au moins|at least|ship)"
)